# Train the V1 representation model

This CPU walkthrough reads complete variable-length files from the persisted train and validation shards. It never generates data in the notebook.

Set `V1_DATA_ROOT` to a generated dataset root, or use the stable default `data/generated/production`.

In [ ]:
from itertools import islice
import json
import os
import tempfile
from pathlib import Path
import sys
for candidate in (Path.cwd() / 'src', Path.cwd().parent / 'src'):
    if (candidate / 'representation').is_dir():
        sys.path.insert(0, str(candidate))
        break
from representation import V1Config
from representation.checkpoint import save_checkpoint
from representation.data import FileDataset, collate_variable_files
from representation.inference import NormalReferenceBank, RepresentationInference
from representation.model import V1RepresentationModel
from representation.trainer import RepresentationTrainer
from synth.config import PatchConfig
from synth.patchify import Patchifier

In [ ]:
configured_root = Path(os.environ.get('V1_DATA_ROOT', 'data/generated/production')).expanduser()
repo_root = Path.cwd()
if not (repo_root / 'src' / 'representation').is_dir():
    repo_root = repo_root.parent
DATA_ROOT = configured_root if configured_root.is_absolute() else repo_root / configured_root
MANIFEST_PATH = DATA_ROOT / 'manifest.json'
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"V1 dataset manifest not found at {MANIFEST_PATH}. Run uv run python -m synth.cli --output {DATA_ROOT} first.")
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
required_splits = ('train', 'val', 'test')
splits = manifest.get('splits')
if not isinstance(splits, dict):
    raise RuntimeError(f"V1 dataset manifest at {MANIFEST_PATH} has no split mapping; regenerate with uv run python -m synth.cli.")
missing_splits = [name for name in required_splits if name not in splits]
if missing_splits:
    raise RuntimeError(f"V1 dataset at {DATA_ROOT} is missing required splits: {', '.join(missing_splits)}. Regenerate with uv run python -m synth.cli.")
def load_split(name: str, limit: int = 4):
    entry = splits[name]
    if not isinstance(entry, dict) or entry.get('status') != 'complete':
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is not complete; rerun uv run python -m synth.cli --output {DATA_ROOT} --resume.")
    samples = list(islice(FileDataset(DATA_ROOT, split=name), limit))
    if not samples:
        raise RuntimeError(f"V1 dataset split {name!r} at {DATA_ROOT} is empty.")
    return samples
train_samples = load_split('train')
val_samples = load_split('val')
print('dataset root', DATA_ROOT, 'manifest counts', manifest['counts'])
print('train IDs', [sample.file_id for sample in train_samples], 'val IDs', [sample.file_id for sample in val_samples])

In [ ]:
cfg = V1Config(n_channels=train_samples[0].C, patch_size=32, stride=16, d_model=8, sequence_layers=1, attention_heads=2, dropout=0.0, contrastive_ramp_steps=2)
patchifier = Patchifier(PatchConfig(patch_size=32, stride=16, pad_end=True))
train_batch = collate_variable_files(train_samples, patchifier, masking_config=cfg, masking_seed=3)
val_batch = collate_variable_files(val_samples, patchifier, masking_config=cfg, masking_seed=4)
print('train signals', tuple(train_batch['signals'].shape), 'validation signals', tuple(val_batch['signals'].shape), 'mask composition', train_batch['mask_composition'])

In [ ]:
model = V1RepresentationModel(cfg, patchifier=patchifier)
trainer = RepresentationTrainer(model, seed=cfg.seed, max_grad_norm=0.5)
history = trainer.fit([train_batch], epochs=1, validation_batches=[val_batch])
print('history', history)
checkpoint_path = Path(tempfile.mkdtemp()) / 'v1-notebook-checkpoint.pt'
save_checkpoint(checkpoint_path, model, optimizer=trainer.optimizer, step=trainer.step)
print('checkpoint', checkpoint_path, 'step', trainer.step)

In [ ]:
model.eval()
reference_output = model(train_batch)
bank = NormalReferenceBank(k=min(2, len(train_samples))).fit(reference_output['file_embedding'])
inference = RepresentationInference(model, bank, patchifier, masking_config=cfg)
scores = inference.score_batch(val_batch)
print('normal train reference rows', bank.embeddings.shape[0], 'lambda', history[-1]['lambda'])
print('validation S_pred', scores['S_pred'].tolist(), 'S_pop', scores['S_pop'].tolist(), 'timestep localization', scores['timestep_scores'][0].tolist())